# Movie Recommendation Bot with Memory (#2.3)

취향(좋아하는 장르)과 **이미 본 영화**를 기억하는 영화 추천 챗봇. **TODO를 직접 채워라.** 막히면 핀포인트 힌트 참고.

- 모델: `gpt-4o-mini`
- 메모리 = 대화를 전부 `messages` 리스트에 누적 → 매 호출마다 통째로 전달
- 함정 1개: 매 턴 **assistant 응답도 append** 해야 기억이 산다 (user만 넣으면 안 됨)

## 0. 준비 — import & 클라이언트

In [11]:
# TODO: os, openai import + dotenv 의 load_dotenv
# TODO: load_dotenv() 로 .env 의 OPENAI_API_KEY 로딩
# TODO: 키 잡혔는지 bool 로만 확인 (값 전체는 찍지 말 것)
# TODO: client = openai.OpenAI(...)   (movie-agent 와 동일)
import os, json, openai
from dotenv import load_dotenv

load_dotenv()
key = os.getenv("OPENAI_API_KEY")
print("key loaded:", bool(key))      

key loaded: True


## 1. 호출 헬퍼 `call_ai(messages)`

단일 질문이 아니라 **messages 리스트 전체**를 받아 모델에 넘긴다. 이게 메모리의 핵심 — 매번 히스토리 통째로.

In [12]:

client = openai.OpenAI(api_key=key)

def call_ai(messages):
        response = client.chat.completions.create(model="gpt-4o-mini", messages=messages)
        return response.choices[0].message.content


## 2. 시스템 프롬프트 + 메모리 초기화

메모리 리스트의 **0번 자리에 system 메시지**를 넣어 봇의 역할/규칙을 박는다.

- 역할: 영화 추천 전문가
- 규칙: 사용자가 말한 **좋아하는 장르**에 맞춰 추천
- 규칙: **이미 봤다고 한 영화는 추천 금지**
- 규칙: 한국어로 자연스럽게

In [13]:
# TODO: SYSTEM_PROMPT = """...위 규칙 4개..."""
# TODO: messages = [ {"role": "system", "content": SYSTEM_PROMPT} ]
SYSTEM_PROMPT = """너는 영화 추천 전문가다.
- 사용자가 말한 좋아하는 장르에 맞춰 추천한다.
- 사용자가 이미 봤다고 한 영화는 절대 다시 추천하지 않는다.
- 한국어로 자연스럽게 답한다."""

messages = [ {"role": "system", "content": SYSTEM_PROMPT} ]


## 3. 한 턴 보내는 헬퍼 `chat(user_input)`

한 턴 = 4단계. 이 함수가 매번 메모리에 양쪽을 쌓는다:
1. `messages.append({"role": "user", "content": user_input})`
2. `call_ai(messages)` → AI 답변
3. `messages.append({"role": "assistant", "content": ...})`  ← 빼먹으면 기억 안 됨
4. user/AI 출력하고 답변 return

In [14]:
# TODO: def chat(user_input):
#         (위 1~4 단계 구현)
def chat(user_input):
  messages.append({"role": "user", "content": user_input})
  ai_message = call_ai(messages) 
  messages.append({"role": "assistant", "content": ai_message })
  print(f"🙋 {user_input}")
  print(f"🤖 {ai_message}")
  return ai_message


## 4. 4턴 시나리오 실행 (제출용 — 출력 찍힌 상태로 커밋)

In [15]:
# TODO: chat("나는 SF 영화를 좋아해")
chat("나는 SF 영화를 좋아해")


🙋 나는 SF 영화를 좋아해
🤖 SF 영화를 좋아하신다면, 다음 몇 편을 추천해 드릴게요! 

1. **"인터스텔라"** - 크리스토퍼 놀란 감독의 작품으로, 우주 탐사를 통한 인류의 미래와 사랑에 대한 이야기를 그렸습니다.
   
2. **"코쿠니"** - 인간의 기억을 저장하고 재생할 수 있는 기술을 다룬 스릴러로, 인간성과 테크놀로지의 경계를 탐구합니다.
   
3. **"블레이드 러너 2049"** - 원작 "블레이드 러너"의 후속작으로, 인공지능과 인간의 존재에 대한 질문을 던지는 매력적인 시각적 경험을 제공합니다.

4. **"주피터 어센딩"** - 우주의 왕국과 더불어 주인공의 운명을 쫓는 이야기를 그린 액션 SF 영화입니다.

이 중에서 마음에 드는 영화가 있으면 좋겠어요! 다른 장르나 더 많은 추천이 필요하시면 말씀해 주세요.


'SF 영화를 좋아하신다면, 다음 몇 편을 추천해 드릴게요! \n\n1. **"인터스텔라"** - 크리스토퍼 놀란 감독의 작품으로, 우주 탐사를 통한 인류의 미래와 사랑에 대한 이야기를 그렸습니다.\n   \n2. **"코쿠니"** - 인간의 기억을 저장하고 재생할 수 있는 기술을 다룬 스릴러로, 인간성과 테크놀로지의 경계를 탐구합니다.\n   \n3. **"블레이드 러너 2049"** - 원작 "블레이드 러너"의 후속작으로, 인공지능과 인간의 존재에 대한 질문을 던지는 매력적인 시각적 경험을 제공합니다.\n\n4. **"주피터 어센딩"** - 우주의 왕국과 더불어 주인공의 운명을 쫓는 이야기를 그린 액션 SF 영화입니다.\n\n이 중에서 마음에 드는 영화가 있으면 좋겠어요! 다른 장르나 더 많은 추천이 필요하시면 말씀해 주세요.'

In [16]:
# TODO: chat("인셉션이랑 인터스텔라는 이미 봤어")
chat("인셉션이랑 인터스텔라는 이미 봤어")

🙋 인셉션이랑 인터스텔라는 이미 봤어
🤖 그렇다면 다른 SF 영화를 추천해 드릴게요!

1. **"소스 코드"** - 잭 기븐이라는 군인이 열차 폭탄 테러를 막기 위해 동일한 8분을 반복적으로 되풀이하는 이야기입니다. 긴장감 넘치는 스릴러 요소가 돋보입니다.

2. **"테넷"** - 크리스토퍼 놀란 감독의 또 다른 작품으로, 시간의 흐름을 거꾸로 하는 독창적인 스토리라인이 특징입니다. 복잡한 이야기 구조가 매력적입니다.

3. **"엑스 마키나"** - 인공지능과 인간의 관계를 심도 있게 탐구하는 영화로, 강력한 스릴과 철학적 질문이 발생합니다.

4. **"그래비티"** - 우주에서의 생존을 그린 긴박감 넘치는 이야기로, 시각적으로도 놀라움을 줍니다.

이 중에서 관심 가는 영화가 있으면 좋겠네요! 추가 추천이 필요하면 언제든지 말씀해 주세요.


'그렇다면 다른 SF 영화를 추천해 드릴게요!\n\n1. **"소스 코드"** - 잭 기븐이라는 군인이 열차 폭탄 테러를 막기 위해 동일한 8분을 반복적으로 되풀이하는 이야기입니다. 긴장감 넘치는 스릴러 요소가 돋보입니다.\n\n2. **"테넷"** - 크리스토퍼 놀란 감독의 또 다른 작품으로, 시간의 흐름을 거꾸로 하는 독창적인 스토리라인이 특징입니다. 복잡한 이야기 구조가 매력적입니다.\n\n3. **"엑스 마키나"** - 인공지능과 인간의 관계를 심도 있게 탐구하는 영화로, 강력한 스릴과 철학적 질문이 발생합니다.\n\n4. **"그래비티"** - 우주에서의 생존을 그린 긴박감 넘치는 이야기로, 시각적으로도 놀라움을 줍니다.\n\n이 중에서 관심 가는 영화가 있으면 좋겠네요! 추가 추천이 필요하면 언제든지 말씀해 주세요.'

In [17]:
# TODO: chat("오늘 밤에 뭐 볼지 추천해 줄래?")   # → SF + 안 본 영화로 추천되는지 확인
chat("오늘 밤에 뭐 볼지 추천해 줄래?")

🙋 오늘 밤에 뭐 볼지 추천해 줄래?
🤖 오늘 밤에 볼 만한 SF 영화로 **"스페이스 오디세이 2001"**을 추천해 드립니다. 클래식한 작품으로, 인류의 기원과 미래를 탐구하는 시각적 아름다움이 인상적입니다. 

또한, **"클로버필드"**도 좋은 선택입니다. 파국적인 상황에서의 인간의 생존을 다룬 재난 영화로, 긴장감 넘치는 스토리 전개가 매력적입니다.

이 두 편 중에서도 흥미로운 영화가 있다면 꼭 즐겨 보세요! 혹시 다른 장르나 더 많은 추천이 필요하시면 언제든지 말씀해 주세요.


'오늘 밤에 볼 만한 SF 영화로 **"스페이스 오디세이 2001"**을 추천해 드립니다. 클래식한 작품으로, 인류의 기원과 미래를 탐구하는 시각적 아름다움이 인상적입니다. \n\n또한, **"클로버필드"**도 좋은 선택입니다. 파국적인 상황에서의 인간의 생존을 다룬 재난 영화로, 긴장감 넘치는 스토리 전개가 매력적입니다.\n\n이 두 편 중에서도 흥미로운 영화가 있다면 꼭 즐겨 보세요! 혹시 다른 장르나 더 많은 추천이 필요하시면 언제든지 말씀해 주세요.'

In [18]:
# TODO: chat("내가 좋아하는 장르랑 이미 본 영화가 뭐라고 했지?")   # → 메모리로 회상되는지 확인
chat("내가 좋아하는 장르랑 이미 본 영화가 뭐라고 했지?")

🙋 내가 좋아하는 장르랑 이미 본 영화가 뭐라고 했지?
🤖 당신은 SF 영화를 좋아한다고 하셨고, "인셉션"과 "인터스텔라"는 이미 보셨다고 하셨습니다. 이 정보를 바탕으로 추천을 드렸죠. 혹시 더 추천을 원하신다면 알려주세요!


'당신은 SF 영화를 좋아한다고 하셨고, "인셉션"과 "인터스텔라"는 이미 보셨다고 하셨습니다. 이 정보를 바탕으로 추천을 드렸죠. 혹시 더 추천을 원하신다면 알려주세요!'

## 5. (선택) 메모리 확인

`messages` 를 출력해 system → user → assistant 가 순서대로 쌓였는지 눈으로 확인.

In [19]:
# TODO: messages 리스트를 보기 좋게 출력 (role 별로)
for m in messages:
    print(f'[{m["role"]}] {m["content"][:60]}...')

[system] 너는 영화 추천 전문가다.
- 사용자가 말한 좋아하는 장르에 맞춰 추천한다.
- 사용자가 이미 봤다고 한 영...
[user] 나는 SF 영화를 좋아해...
[assistant] SF 영화를 좋아하신다면, 다음 몇 편을 추천해 드릴게요! 

1. **"인터스텔라"** - 크리스토퍼 놀란...
[user] 인셉션이랑 인터스텔라는 이미 봤어...
[assistant] 그렇다면 다른 SF 영화를 추천해 드릴게요!

1. **"소스 코드"** - 잭 기븐이라는 군인이 열차 폭탄...
[user] 오늘 밤에 뭐 볼지 추천해 줄래?...
[assistant] 오늘 밤에 볼 만한 SF 영화로 **"스페이스 오디세이 2001"**을 추천해 드립니다. 클래식한 작품으로,...
[user] 내가 좋아하는 장르랑 이미 본 영화가 뭐라고 했지?...
[assistant] 당신은 SF 영화를 좋아한다고 하셨고, "인셉션"과 "인터스텔라"는 이미 보셨다고 하셨습니다. 이 정보를 바...
